In [2]:
# 모델 호출

import os
from dotenv import load_dotenv
from langchain.chat_models import init_chat_model
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

load_dotenv()

model = init_chat_model(
    model_provider="openai",
    model="gpt-6-luna",
)

parser = StrOutputParser()

prompt = ChatPromptTemplate.from_messages([
    ("system", "당신은 질문에 솔직하고 정확하게 답변하는 비서입니다."), # (역할, 내용 템플릿)
    ("user", "{questions}")  # {} 안은 실행할 때 채워질 변수
])

chain = prompt | model | parser

question_text = "2026년 한양대학교 컴퓨터공학과 1학기 축제 개최일과 초청 가수를 알려줘."
response = chain.invoke({"questions": question_text})

print("=== [단순 LLM 호출 결과] ===")
print(response)

=== [단순 LLM 호출 결과] ===
죄송하지만 실시간 웹 확인을 할 수 없어 **2026년 한양대학교 컴퓨터공학과 1학기 축제의 개최일과 초청 가수**를 공식 자료로 확인해 드리기는 어렵습니다. 확인되지 않은 날짜나 가수 이름을 추측해 안내하진 않겠습니다.

한양대는 서울캠퍼스와 ERICA캠퍼스 행사가 다를 수 있으니, 해당 학과 학생회와 각 캠퍼스 총학생회의 공식 공지·SNS에서 확인해 주세요.


In [3]:
rag_prompt = ChatPromptTemplate.from_messages([
    (
        "system",
        "당신은 제공된 [참고 문서]의 내용에만 근거하여 정확하게 답변하는 전문 튜터입니다. \n"
        "문서에 나와 있지 않은 내용은 절대로 지어내지 말고 모른다고 답변하세요. \n\n"
        "[참고 문서]:\n{context}"
    ),
    ("user", "{question}")
])

rag_chain = rag_prompt | model | parser

retrieved_context = """
[공지] 2026학년도 스파르타대학교 컴퓨터공학과 봄 축제 안내
- 일시: 2026년 5월 14일(목) ~ 5월 15일(금)
- 장소: IT융합관 앞 잔디광장
- 초청 아티스트: 뉴진스, 악동뮤지션
- 유의사항: 학생증 지참 필수
"""

rag_response = rag_chain.invoke({
    "context": retrieved_context,
    "question": "2026년 스파르타대학교 컴퓨터공학과 1학기 축제 개최일과 초청 가수를 알려줘."
})

print("=== [RAG 기반 생성 결과] ===")
print(rag_response)

=== [RAG 기반 생성 결과] ===
축제는 2026년 5월 14일(목)부터 5월 15일(금)까지 열리며, 초청 가수는 뉴진스와 악동뮤지션입니다.
